# Script 7–9 — Feature Retrieval & Model Building (Papermill/Prefect-safe)
This notebook assumes `data/transformed/churn_features.csv` exists.
It trains two models (LogReg, RandomForest), saves pickles under `models/`, and metrics JSONs under `reports/`.

In [1]:
import os, json, joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

features_path = 'data/transformed/churn_features.csv'
assert os.path.exists(features_path), f'Missing {features_path}. Run Tasks 2–6 first.'
df = pd.read_csv(features_path)
print('Loaded:', features_path, 'shape=', df.shape)

Loaded: data/transformed/churn_features.csv shape= (6, 8)


In [2]:
# Prepare data
y = df['churn'].astype(int)
X = df.drop(columns=['churn']).select_dtypes(include=[np.number])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Train/Test sizes:', X_train.shape, X_test.shape)

Train/Test sizes: (4, 6) (2, 6)


In [3]:
# Logistic Regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_s, y_train)
probs = logreg.predict_proba(X_test_s)[:,1]
preds = (probs >= 0.5).astype(int)
metrics_lr = {
    'accuracy': float(accuracy_score(y_test, preds)),
    'precision': float(precision_score(y_test, preds, zero_division=0)),
    'recall': float(recall_score(y_test, preds, zero_division=0)),
    'f1': float(f1_score(y_test, preds, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, probs)) if len(np.unique(y_test))>1 else None
}
joblib.dump({'scaler': scaler, 'model': logreg, 'features': list(X.columns)}, 'models/logreg.pkl')
with open('reports/evaluation_logreg.json','w') as f: json.dump({'model':'logreg','metrics':metrics_lr}, f, indent=2)
print('Saved logreg model + report')

Saved logreg model + report


In [4]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
probs_rf = rf.predict_proba(X_test)[:,1]
preds_rf = (probs_rf >= 0.5).astype(int)
metrics_rf = {
    'accuracy': float(accuracy_score(y_test, preds_rf)),
    'precision': float(precision_score(y_test, preds_rf, zero_division=0)),
    'recall': float(recall_score(y_test, preds_rf, zero_division=0)),
    'f1': float(f1_score(y_test, preds_rf, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, probs_rf)) if len(np.unique(y_test))>1 else None
}
joblib.dump({'model': rf, 'features': list(X.columns)}, 'models/random_forest.pkl')
with open('reports/evaluation_random_forest.json','w') as f: json.dump({'model':'random_forest','metrics':metrics_rf}, f, indent=2)
print('Saved random forest model + report')

Saved random forest model + report


In [5]:
# Save sample retrieval for Point 7 proof
df.head(10).to_csv('reports/retrieved_features_sample.csv', index=False)
print('Saved reports/retrieved_features_sample.csv')

Saved reports/retrieved_features_sample.csv


In [6]:
# Summary
import glob, json
artifacts = {
    'models': sorted(glob.glob('models/*.pkl')),
    'reports': sorted(glob.glob('reports/*.json')),
    'retrieval_sample': 'reports/retrieved_features_sample.csv'
}
print(json.dumps(artifacts, indent=2))

{
  "models": [
    "models/logreg.pkl",
    "models/random_forest.pkl"
  ],
  "reports": [
    "reports/evaluation_logreg.json",
    "reports/evaluation_random_forest.json"
  ],
  "retrieval_sample": "reports/retrieved_features_sample.csv"
}
